# Aurora Supply: dados → previsão → MLflow
Dados fictícios CC0, ano2025. O modelo aprende tendência e sazonalidade. O último mês é holdout.
Um candidato pode perder para baseline; ele é rejeitado, sem inventar métricas.


In [ ]:
from pathlib import Path
import sys, json
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from science import generate, train_sku, bundle, PRODUCTS
rows = generate(repo / "data")
print(len(rows), "linhas sintéticas")


In [ ]:
models = [train_sku(rows, p["sku"]) for p in PRODUCTS]
[(m["sku"], round(m["candidate_mae"], 3), round(m["baseline_mae"], 3), m["model_type"]) for m in models]


In [ ]:
result = bundle(models, rows)
print(json.dumps(result["forecasts"][0], indent=2, ensure_ascii=False))


In [ ]:
%pip install "mlflow[kubernetes]==3.14.0" "boto3==1.43.18"


In [ ]:
from science import publish
# Grava experimento real e artefato no S3; falha se integração não funcionar.
published = publish(result)
print(published["mlflow_run_id"])
